In [ ]:
import os
import numpy as np
import pandas as pd
from utils import configs, callbacks
from metrics import F1Score, report
import tensorflow as tf
from models.mbcnn import mbcnn
from losses import losses
from utils import dataloader as dl, plot, inference

In [ ]:
cfg = configs.load_configs('config.yaml')

city = 'salvador_brazil'
dir = os.path.join(cfg.PROCESSED_DATA_PATH, city)
inputs = ".".join(cfg.DATASET)
input_shapes = {input: cfg.IN_SHAPE[input] for input in cfg.DATASET}
h, w, _ = cfg.IN_SHAPE[cfg.DATASET[0]]


In [ ]:
train_dataset = dl.MBCNNDataset(
    os.path.join(dir, "train"),
    target_size=cfg.PATCH_SIZE
)
train_loader = dl.mbcnn_loader(train_dataset, batch_size=cfg.BATCH_SIZE, shuffle=True)

val_dataset = dl.MBCNNDataset(
    os.path.join(dir, "val"),
    target_size=cfg.PATCH_SIZE
)
val_loader = dl.mbcnn_loader(val_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False)

In [ ]:
for batch in train_loader:
    s2, rf = batch
    print(f"Inputs Batch shape: {s2[0].shape}, {s2[1].shape}")
    print(f"Targets Batch shape: {rf.shape}")
    break

# check the size of the dataset
print(f"Number of training samples: {len(train_dataset)}")
print(f"Number of validation samples: {len(val_dataset)}")

In [ ]:
# Calculate class weights from training dataset
_, class_weight_list = train_dataset.compute_class_weights()
print(f"Class weights: {class_weight_list}")

In [ ]:
# create iterator for training dataloader
train_iter = iter(train_loader)

In [ ]:
plot.plot_samples(train_iter, batch_size=cfg.BATCH_SIZE)

In [ ]:
import segmentation_models as sm
dice_loss = sm.losses.DiceLoss(class_weights=class_weight_list) 
focal_loss = sm.losses.CategoricalFocalLoss()
t_loss =  dice_loss + (2 * focal_loss)

In [ ]:
model = mbcnn(cfg.N_CLASSES, input_shapes, dropout_rate=0.0, batch_norm=True, drop_train=False)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=cfg.LR),
    loss=t_loss,
    metrics=[F1Score(name='f1')]
)

In [ ]:
callbacks = callbacks.build_callbacks(city, inputs, model, train_loader, cfg)

model.fit(train_loader,
        epochs=cfg.N_EPOCHS,
        callbacks=callbacks,
        validation_data=val_loader,
        verbose=0)


In [ ]:
# # ## Plot training log
history = pd.read_csv(os.path.join(cfg.LOG_PATH, f"{city}.{inputs.lower()}.{model.name}.log.csv"))
plot.plot_log(history)

In [ ]:
model = mbcnn(CL=cfg.N_CLASSES, input_shapes=input_shapes, dropout_rate=0.2, batch_norm=True, drop_train=False)
weight = (os.path.join(cfg.CHECKPOINT_PATH, f"{city}.{inputs.lower()}.{model.name}.weights.h5"))
model.load_weights(weight)

In [ ]:
# load test dataset
test_dataset = dl.MBCNNDataset(
    os.path.join(dir, "test"),
    target_size=cfg.PATCH_SIZE
)
test_loader = dl.mbcnn_loader(test_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False)

In [ ]:
# check the size of the dataset
print(f"Number of test samples: {len(test_dataset)}")

In [ ]:
class_report, cm, iou_values, mean_iou, fwiou = report(test_loader, model)

print("Classification Report:\n" + str(class_report))
print(f"IoU per Class:\n{iou_values}")
print(f"Mean IoU: {mean_iou:.4f}")
print(f"Frequency Weighted IoU: {fwiou:.4f}")

In [ ]:
# show confusion matrix
print("Confusion Matrix:")
print(cm)

In [ ]:
test_iter = iter(test_loader)

In [ ]:
plot.plot_prediction(test_iter, model, cfg.BATCH_SIZE)

In [ ]:
# run inference on a single image
s2 = os.path.join(cfg.RAW_DATA_PATH, 'sentinel', city, "s2.tif")
rf = os.path.join(cfg.RAW_DATA_PATH, 'buildings/density', city, "rf.tif")

image_sources = {
    "s2": s2,
    "rf": rf
}


In [ ]:
inference.inference_mbcnn(
    model=model,
    image_sources=image_sources,
    save_path=cfg.PREDICTION_PATH,
    aoi_path=os.path.join(cfg.AOI_PATH, f"{city}_aoi.geojson"),
    batch_size=cfg.BATCH_SIZE,
    stride_ratio=cfg.STRIDE_RATIO
)